# LFW Demo Enrollment — Load identities vào VectorDB cho demo webcam

Notebook này load **N identity từ LFW** (không phải từ thư mục gallery/) vào VectorDB để
phục vụ test nhận diện realtime trên webcam.

Với mỗi **enrolled identity**:
- **Gallery images** → pipeline (Detect → Align → Embed) → mean embedding → `db.upsert(overwrite=True)`.
- **Test images** (1–2 ảnh **không được dùng** trong gallery) → **tự động copy vào thư mục** `lfw_demo_test_images/<name>/` theo từng identity.
  Dùng để test replay attack (hiển thị trên iPad trước webcam) hoặc in ra giấy test print attack.

Với mỗi **impostor identity** (không được enroll vào DB):
- 1–2 ảnh được **tự động copy vào thư mục** `lfw_demo_impostors/<name>/` theo từng identity — dùng để test hệ thống có từ chối
  người lạ (UNKNOWN) đúng không.

**Chạy lại notebook sẽ overwrite** toàn bộ record cũ (cùng `user_id = demo_lfw_<name>`).


In [11]:
# =========================
# 1. PARAMETERS
# =========================

from pathlib import Path

# ── Số identity cần load ──────────────────────────────────────────────
NUM_DEMO_IDENTITIES = 30        # Thay đổi từ 20–50 tuỳ nhu cầu demo

# ── Số ảnh dùng để build gallery embedding (khuyến nghị = 3) ─────────────
GALLERY_IMAGES_PER_ID = 3

# ── Số ảnh KHÔNG dùng cho gallery, giữ lại làm test set ──────────────
# Ảnh này sẽ được copy vào thư mục lfw_demo_test_images/<name>/
TEST_IMAGES_PER_ID = 2

# ── Impostor: identity KHÔNG được enroll vào DB ────────────────────
# Ảnh này sẽ được copy vào thư mục lfw_demo_impostors/<name>/
NUM_IMPOSTOR_PREVIEW = 10       # Số impostor identity được chọn
IMPOSTOR_IMAGES_PER_ID = 2      # Số ảnh mỗi impostor copy vào thư mục

# Mỗi identity cần ít nhất bao nhiêu ảnh để đủ điều kiện enrolled
MIN_IMAGES_PER_ID = GALLERY_IMAGES_PER_ID + TEST_IMAGES_PER_ID  # = 5

RANDOM_SEED = 42

# Prefix namespace VectorDB — tách biệt hoàn toàn với gallery/ thật
VECTORDB_NAMESPACE = "demo_lfw"

# Output files và thư mục lưu ảnh (nằm cùng thư mục notebook)
MANIFEST_CSV     = "lfw_demo_manifest.csv"        # Danh sách user đã upsert
TEST_IMAGES_DIR  = "lfw_demo_test_images"         # Thư mục chứa ảnh test theo từng identity
IMPOSTOR_DIR     = "lfw_demo_impostors"           # Thư mục chứa ảnh impostor theo từng identity

print(f"NUM_DEMO_IDENTITIES    = {NUM_DEMO_IDENTITIES}")
print(f"GALLERY_IMAGES_PER_ID  = {GALLERY_IMAGES_PER_ID}")
print(f"TEST_IMAGES_PER_ID     = {TEST_IMAGES_PER_ID}")
print(f"NUM_IMPOSTOR_PREVIEW   = {NUM_IMPOSTOR_PREVIEW}")
print(f"IMPOSTOR_IMAGES_PER_ID = {IMPOSTOR_IMAGES_PER_ID}")
print(f"MIN_IMAGES_PER_ID      = {MIN_IMAGES_PER_ID}")
print(f"Namespace prefix       = {VECTORDB_NAMESPACE}_<name>")
print(f"Test images folder     = {TEST_IMAGES_DIR}/<name>/")
print(f"Impostor folder        = {IMPOSTOR_DIR}/<name>/")


NUM_DEMO_IDENTITIES    = 30
GALLERY_IMAGES_PER_ID  = 3
TEST_IMAGES_PER_ID     = 2
NUM_IMPOSTOR_PREVIEW   = 10
IMPOSTOR_IMAGES_PER_ID = 2
MIN_IMAGES_PER_ID      = 5
Namespace prefix       = demo_lfw_<name>
Test images folder     = lfw_demo_test_images/<name>/
Impostor folder        = lfw_demo_impostors/<name>/


In [12]:
# =========================
# 2. PROJECT / DATASET PATH
# =========================

import sys
import os
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "config.py").exists() and (p / "data").exists():
            return p
    raise FileNotFoundError("Không tìm thấy project root.")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LFW_ROOT = PROJECT_ROOT / "data" / "lfw"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"LFW_ROOT     : {LFW_ROOT}")

if not LFW_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy LFW dataset tại {LFW_ROOT}")


PROJECT_ROOT : /Users/coding/PBL6/face_auth
LFW_ROOT     : /Users/coding/PBL6/face_auth/data/lfw


In [13]:
# =========================
# 3. INVENTORY + ELIGIBLE IDENTITIES
# =========================

identity_rows = []
for d in sorted(LFW_ROOT.iterdir()):
    if not d.is_dir():
        continue
    images = sorted([p for p in d.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    if images:
        identity_rows.append({"identity": d.name, "image_count": len(images), "dir": str(d)})

inventory = pd.DataFrame(identity_rows).sort_values(
    ["image_count", "identity"], ascending=[False, True]
).reset_index(drop=True)

eligible = inventory[inventory["image_count"] >= MIN_IMAGES_PER_ID].copy()

print(f"Total LFW identities  : {len(inventory):,}")
print(f"Eligible (>= {MIN_IMAGES_PER_ID} images): {len(eligible):,}")

if len(eligible) < NUM_DEMO_IDENTITIES:
    raise ValueError(
        f"Chỉ có {len(eligible)} identity đủ {MIN_IMAGES_PER_ID} ảnh. "
        f"Giảm MIN_IMAGES_PER_ID hoặc NUM_DEMO_IDENTITIES."
    )


Total LFW identities  : 5,749
Eligible (>= 5 images): 423


In [14]:
# =========================
# 4. SELECT + SPLIT IMAGES
# =========================

rng = np.random.default_rng(RANDOM_SEED)

names_pool = eligible["identity"].tolist()
rng.shuffle(names_pool)
selected_names = sorted(names_pool[:NUM_DEMO_IDENTITIES])

gallery_paths = {}
test_paths = {}

for name in selected_names:
    identity_dir = LFW_ROOT / name
    images = sorted([p for p in identity_dir.iterdir()
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    idx = rng.permutation(len(images))

    gallery = [images[int(i)] for i in idx[:GALLERY_IMAGES_PER_ID]]
    test    = [images[int(i)] for i in idx[GALLERY_IMAGES_PER_ID:GALLERY_IMAGES_PER_ID + TEST_IMAGES_PER_ID]]

    gallery_paths[name] = gallery
    test_paths[name]    = test

print(f"Selected identities: {len(selected_names)}")
print(f"Sample: {selected_names[:5]}")
print(f"Gallery images per id: {GALLERY_IMAGES_PER_ID}")
print(f"Test images per id   : {TEST_IMAGES_PER_ID}")


Selected identities: 30
Sample: ['Allyson_Felix', 'Ana_Guevara', 'Ashanti', 'Bertie_Ahern', 'Britney_Spears']
Gallery images per id: 3
Test images per id   : 2


In [15]:
# =========================
# 5. LOAD PROJECT PIPELINE
# =========================

import cv2
import config
from detection.detector import FaceDetector
from recognition.embedder import FaceEmbedder
from alignment.aligner import get_input_face
from enrollment.enroll import build_identity_embedding
from database.vector_db import VectorDB

detector = FaceDetector(
    config.MODEL_PACK_NAME,
    ctx_id=config.MODEL_CTX_ID,
    det_size=config.DETECTOR_DET_SIZE,
    conf_thresh=config.DETECTOR_CONF_THRESH,
)
embedder = FaceEmbedder(config.MODEL_PACK_NAME, ctx_id=config.MODEL_CTX_ID)
db = VectorDB(
    conninfo=config.DB_CONN_INFO,
    min_size=config.DB_POOL_MIN_SIZE,
    max_size=config.DB_POOL_MAX_SIZE,
)

print(f"Pipeline ready | MATCH_THRESHOLD = {config.MATCH_THRESHOLD}")


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /Users/3t/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /Users/3t/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/3t/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /Users/3t/.insightface/models/buffalo_s/genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /Users/3t/.insightface/models/buffalo_s/w600k_mbf.onnx recognition
set det-size: (640, 640)
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /Us

In [16]:
# =========================
# 6. EMBED FUNCTION
# =========================

from typing import Optional

def embed_image(path: Path) -> Optional[np.ndarray]:
    img = cv2.imread(str(path))
    if img is None:
        return None
    detections = detector.detect(img)
    if not detections:
        return None
    det = max(detections, key=lambda d: d.get("score", 0.0))
    aligned = get_input_face(img, det["bbox"], det.get("landmarks"), embedder.input_size)
    if aligned is None:
        return None
    emb = embedder.embed_aligned(aligned)
    if emb is None:
        return None
    emb = np.asarray(emb, dtype=np.float32).flatten()
    norm = np.linalg.norm(emb)
    return emb / norm if norm > 0 else None


In [17]:
# =========================
# 7. ENROLL TO VECTORDB
# =========================

import re
from datetime import datetime, timezone
from tqdm.auto import tqdm

def make_user_id(name: str) -> str:
    # demo_lfw_<name> — tách hoàn toàn khỏi gallery/ thật và benchmark_lfw_
    return f"{VECTORDB_NAMESPACE}_" + re.sub(r"[^a-zA-Z0-9_-]+", "_", name)

manifest_rows = []
skipped = []

for name in tqdm(selected_names, desc="Enrolling"):
    user_id = make_user_id(name)

    # Embed gallery images
    raw_embs = []
    for p in gallery_paths[name]:
        e = embed_image(p)
        if e is not None:
            raw_embs.append(e)

    if not raw_embs:
        skipped.append(name)
        print(f"[SKIP] {name}: không embed được ảnh nào từ gallery")
        continue

    # Build mean embedding (với outlier filtering)
    try:
        mean_emb, warnings, valid_embs = build_identity_embedding(
            raw_embs, outlier_threshold=config.RECOMMENDED_GALLERY_SIZE and 0.35
        )
    except Exception as e:
        skipped.append(name)
        print(f"[SKIP] {name}: {e}")
        continue

    for w in warnings:
        print(f"  [{name}] {w}")

    # Upsert vào VectorDB (overwrite=True để re-run không tạo duplicate)
    person_id, is_new, is_ignored = db.upsert(
        user_id=user_id,
        name=name,
        individual_embeddings=valid_embs,
        mean_embedding=mean_emb,
        overwrite=True,
        save_individuals=True,
    )

    manifest_rows.append({
        "user_id":            person_id,
        "name":               name,
        "gallery_image_count": len(valid_embs),
        "gallery_images":     ";".join(p.name for p in gallery_paths[name]),
        "test_image_count":   len(test_paths[name]),
        "is_new":             bool(is_new),
        "enrolled_at_utc":    datetime.now(timezone.utc).isoformat(),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(NOTEBOOK_DIR / MANIFEST_CSV, index=False, encoding="utf-8-sig")
print(f"\nEnrolled : {len(manifest_rows)} identities")
print(f"Skipped  : {len(skipped)}")
print(f"Manifest : {NOTEBOOK_DIR / MANIFEST_CSV}")
display(manifest_df.head(10))


Enrolling:   0%|          | 0/30 [00:00<?, ?it/s]


Enrolled : 30 identities
Skipped  : 0
Manifest : /Users/coding/PBL6/face_auth/evaluation/lfw_demo_manifest.csv


,user_id,name,gallery_image_count,gallery_images,test_image_count,is_new,enrolled_at_utc
0,demo_lfw_Allyson_Felix,Allyson_Felix,3,Allyson_Felix_0005.jpg;Allyson_Felix_0004.jpg;...,2,False,2026-09-02T15:11:31.098692+00:00
1,demo_lfw_Ana_Guevara,Ana_Guevara,3,Ana_Guevara_0005.jpg;Ana_Guevara_0001.jpg;Ana_...,2,False,2026-09-02T15:11:31.158052+00:00
2,demo_lfw_Ashanti,Ashanti,3,Ashanti_0002.jpg;Ashanti_0004.jpg;Ashanti_0005...,2,False,2026-09-02T15:11:31.216849+00:00
3,demo_lfw_Bertie_Ahern,Bertie_Ahern,3,Bertie_Ahern_0003.jpg;Bertie_Ahern_0005.jpg;Be...,2,False,2026-09-02T15:11:31.272978+00:00
4,demo_lfw_Britney_Spears,Britney_Spears,3,Britney_Spears_0004.jpg;Britney_Spears_0007.jp...,2,False,2026-09-02T15:11:31.330539+00:00
5,demo_lfw_Christine_Baumgartner,Christine_Baumgartner,3,Christine_Baumgartner_0002.jpg;Christine_Baumg...,2,False,2026-09-02T15:11:31.389568+00:00
6,demo_lfw_Claudia_Pechstein,Claudia_Pechstein,3,Claudia_Pechstein_0004.jpg;Claudia_Pechstein_0...,2,False,2026-09-02T15:11:31.446753+00:00
7,demo_lfw_David_Trimble,David_Trimble,3,David_Trimble_0004.jpg;David_Trimble_0001.jpg;...,2,False,2026-09-02T15:11:31.501838+00:00
8,demo_lfw_George_W_Bush,George_W_Bush,2,George_W_Bush_0052.jpg;George_W_Bush_0330.jpg;...,2,False,2026-09-02T15:11:31.551777+00:00
9,demo_lfw_Grant_Hackett,Grant_Hackett,3,Grant_Hackett_0002.jpg;Grant_Hackett_0003.jpg;...,2,False,2026-09-02T15:11:31.608200+00:00


In [18]:
# =========================
# 8. COPY TEST IMAGES INTO FOLDERS BY IDENTITY
# =========================
# Copy ảnh test (KHÔNG dùng trong gallery) vào thư mục con theo từng identity:
#   lfw_demo_test_images/<identity_name>/<image_name>
# Giúp bạn mở trực tiếp trên iPad/điện thoại hoặc in ra để test nhận diện.

import shutil

TEST_FOLDER = NOTEBOOK_DIR / TEST_IMAGES_DIR
TEST_FOLDER.mkdir(parents=True, exist_ok=True)

# Xóa nội dung cũ trước khi copy (để re-run sạch)
for old_d in TEST_FOLDER.iterdir():
    if old_d.is_dir():
        shutil.rmtree(old_d)

enrolled_ids = {row["name"] for row in manifest_rows}
test_manifest_rows = []

for name in selected_names:
    if name not in enrolled_ids:
        continue
    user_id = make_user_id(name)
    identity_test_dir = TEST_FOLDER / name
    identity_test_dir.mkdir(parents=True, exist_ok=True)

    for p in test_paths[name]:
        dest_path = identity_test_dir / p.name
        shutil.copy2(p, dest_path)
        test_manifest_rows.append({
            "user_id":         user_id,
            "name":            name,
            "folder":          str(identity_test_dir),
            "image_name":      p.name,
            "image_path":      str(dest_path),
        })

test_df = pd.DataFrame(test_manifest_rows)
TEST_CSV = TEST_IMAGES_DIR + "_manifest.csv"
test_df.to_csv(NOTEBOOK_DIR / TEST_CSV, index=False, encoding="utf-8-sig")

print(f"Test identities copied : {len(test_df['name'].unique())}")
print(f"Total test images      : {len(test_manifest_rows)}")
print(f"Saved to folder        : {TEST_FOLDER}")
print(f"Manifest CSV           : {NOTEBOOK_DIR / TEST_CSV}")
print()
print("Cách sử dụng ảnh test:")
print("  1. Mở thư mục lfw_demo_test_images/<name>/ để xem các ảnh test của từng người.")
print("  2. Replay attack: Mở ảnh trên iPad/màn hình rồi đưa trước webcam.")
print("  3. Print attack : In ảnh ra giấy rồi đưa trước webcam.")
print(f"  4. app.py sẽ nhận diện [OK] <name> với confidence >= {config.MATCH_THRESHOLD}")
display(test_df.head(10))


Test identities copied : 30
Total test images      : 60
Saved to folder        : /Users/coding/PBL6/face_auth/evaluation/lfw_demo_test_images
Manifest CSV           : /Users/coding/PBL6/face_auth/evaluation/lfw_demo_test_images_manifest.csv

Cách sử dụng ảnh test:
  1. Mở thư mục lfw_demo_test_images/<name>/ để xem các ảnh test của từng người.
  2. Replay attack: Mở ảnh trên iPad/màn hình rồi đưa trước webcam.
  3. Print attack : In ảnh ra giấy rồi đưa trước webcam.
  4. app.py sẽ nhận diện [OK] <name> với confidence >= 0.35


,user_id,name,folder,image_name,image_path
0,demo_lfw_Allyson_Felix,Allyson_Felix,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Allyson_Felix_0001.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
1,demo_lfw_Allyson_Felix,Allyson_Felix,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Allyson_Felix_0002.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
2,demo_lfw_Ana_Guevara,Ana_Guevara,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Ana_Guevara_0002.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
3,demo_lfw_Ana_Guevara,Ana_Guevara,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Ana_Guevara_0004.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
4,demo_lfw_Ashanti,Ashanti,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Ashanti_0001.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
5,demo_lfw_Ashanti,Ashanti,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Ashanti_0003.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
6,demo_lfw_Bertie_Ahern,Bertie_Ahern,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Bertie_Ahern_0004.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
7,demo_lfw_Bertie_Ahern,Bertie_Ahern,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Bertie_Ahern_0002.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
8,demo_lfw_Britney_Spears,Britney_Spears,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Britney_Spears_0005.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...
9,demo_lfw_Britney_Spears,Britney_Spears,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Britney_Spears_0008.jpg,/Users/coding/PBL6/face_auth/evaluation/lfw_de...


In [19]:
# =========================
# 9. VERIFICATION — Kiểm tra lại DB
# =========================

from database.vector_db import decide_identity

# Đếm số user có prefix demo_lfw_ trong DB
with db.pool.connection() as conn:
    rows = conn.execute(
        "SELECT COUNT(DISTINCT user_id) FROM users WHERE user_id LIKE %s",
        (f"{VECTORDB_NAMESPACE}_%",)
    ).fetchone()
    total_in_db = rows[0] if rows else 0

    mean_rows = conn.execute(
        "SELECT COUNT(*) FROM face_embeddings f "
        "JOIN users u ON f.user_id = u.user_id "
        "WHERE u.user_id LIKE %s AND f.is_mean = TRUE",
        (f"{VECTORDB_NAMESPACE}_%",)
    ).fetchone()
    total_mean_vecs = mean_rows[0] if mean_rows else 0

print(f"Users in DB with prefix '{VECTORDB_NAMESPACE}_': {total_in_db}")
print(f"Mean vectors in DB                             : {total_mean_vecs}")
print()
print("✓ Sẵn sàng chạy app.py để test nhận diện realtime trên webcam!")
print(f"  python app.py")
print()
print("Để cleanup toàn bộ demo identities khỏi DB, chạy:")
print(f"  DELETE FROM users WHERE user_id LIKE \'{VECTORDB_NAMESPACE}_%\';")

db.close()


Users in DB with prefix 'demo_lfw_': 30
Mean vectors in DB                             : 30

✓ Sẵn sàng chạy app.py để test nhận diện realtime trên webcam!
  python app.py

Để cleanup toàn bộ demo identities khỏi DB, chạy:
  DELETE FROM users WHERE user_id LIKE 'demo_lfw_%';


In [20]:
# =========================
# 10. IMPOSTOR PREVIEW FOLDER
# =========================
# Chọn các identity KHÔNG enroll vào DB, copy ảnh vào lfw_demo_impostors/<name>/
# Dùng để test hệ thống có trả UNKNOWN đúng cho người lạ.

import shutil

# Pool impostor: eligible nhưng không nằm trong selected_names (đã enroll vào DB)
enrolled_set = set(selected_names)
impostor_pool = [
    name for name in eligible["identity"].tolist()
    if name not in enrolled_set
]

# Shuffle riêng (không ảnh hưởng seed của enrolled)
rng_imp = np.random.default_rng(RANDOM_SEED + 99)
rng_imp.shuffle(impostor_pool)
selected_impostors = impostor_pool[:NUM_IMPOSTOR_PREVIEW]

IMPOSTOR_FOLDER = NOTEBOOK_DIR / IMPOSTOR_DIR
IMPOSTOR_FOLDER.mkdir(parents=True, exist_ok=True)

# Xóa nội dung cũ (để re-run sạch)
for old_d in IMPOSTOR_FOLDER.iterdir():
    if old_d.is_dir():
        shutil.rmtree(old_d)

impostor_manifest_rows = []

for name in selected_impostors:
    identity_dir = LFW_ROOT / name
    images = sorted([p for p in identity_dir.iterdir()
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    # Chọn ngẫu nhiên IMPOSTOR_IMAGES_PER_ID ảnh
    rng_imp.shuffle(images)
    chosen = images[:IMPOSTOR_IMAGES_PER_ID]

    # Tạo thư mục con và copy ảnh
    dest_dir = IMPOSTOR_FOLDER / name
    dest_dir.mkdir(parents=True, exist_ok=True)
    for img_path in chosen:
        shutil.copy2(img_path, dest_dir / img_path.name)
        impostor_manifest_rows.append({
            "name":           name,
            "source_path":    str(img_path),
            "copy_path":      str(dest_dir / img_path.name),
            "image_name":     img_path.name,
        })

impostor_df = pd.DataFrame(impostor_manifest_rows)
IMPOSTOR_CSV = IMPOSTOR_DIR + "_manifest.csv"
impostor_df.to_csv(NOTEBOOK_DIR / IMPOSTOR_CSV, index=False, encoding="utf-8-sig")

print(f"Impostor identities copied : {len(selected_impostors)}")
print(f"Total impostor images      : {len(impostor_manifest_rows)}")
print(f"Saved to folder            : {IMPOSTOR_FOLDER}")
print(f"Manifest CSV               : {NOTEBOOK_DIR / IMPOSTOR_CSV}")
print()
print("Cách sử dụng:")
print("  Mở ảnh trong lfw_demo_impostors/<name>/ trên iPad rồi đưa trước webcam.")
print("  Hệ thống phải trả UNKNOWN (viền đỏ) vì các identity này không có trong DB.")
display(impostor_df)


Impostor identities copied : 10
Total impostor images      : 20
Saved to folder            : /Users/coding/PBL6/face_auth/evaluation/lfw_demo_impostors
Manifest CSV               : /Users/coding/PBL6/face_auth/evaluation/lfw_demo_impostors_manifest.csv

Cách sử dụng:
  Mở ảnh trong lfw_demo_impostors/<name>/ trên iPad rồi đưa trước webcam.
  Hệ thống phải trả UNKNOWN (viền đỏ) vì các identity này không có trong DB.


,name,source_path,copy_path,image_name
0,Mick_Jagger,/Users/coding/PBL6/face_auth/data/lfw/Mick_Jag...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Mick_Jagger_0003.jpg
1,Mick_Jagger,/Users/coding/PBL6/face_auth/data/lfw/Mick_Jag...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Mick_Jagger_0004.jpg
2,George_HW_Bush,/Users/coding/PBL6/face_auth/data/lfw/George_H...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,George_HW_Bush_0004.jpg
3,George_HW_Bush,/Users/coding/PBL6/face_auth/data/lfw/George_H...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,George_HW_Bush_0001.jpg
4,Jacques_Chirac,/Users/coding/PBL6/face_auth/data/lfw/Jacques_...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Jacques_Chirac_0009.jpg
5,Jacques_Chirac,/Users/coding/PBL6/face_auth/data/lfw/Jacques_...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Jacques_Chirac_0034.jpg
6,Vaclav_Havel,/Users/coding/PBL6/face_auth/data/lfw/Vaclav_H...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Vaclav_Havel_0009.jpg
7,Vaclav_Havel,/Users/coding/PBL6/face_auth/data/lfw/Vaclav_H...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Vaclav_Havel_0008.jpg
8,Vicente_Fernandez,/Users/coding/PBL6/face_auth/data/lfw/Vicente_...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Vicente_Fernandez_0003.jpg
9,Vicente_Fernandez,/Users/coding/PBL6/face_auth/data/lfw/Vicente_...,/Users/coding/PBL6/face_auth/evaluation/lfw_de...,Vicente_Fernandez_0005.jpg
